In [1]:
import sys
from pathlib import Path
sys.path.insert(0, "/Users/danilamalafeev/Documents/New project/build-release")

import yabe
import os

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))

In [2]:
import pandas as pd

In [4]:
paths = [
    PROJECT_ROOT + "/data/BTCUSDT-trades-2024-03-05.csv",
    PROJECT_ROOT + "/data/ETHUSDT-trades-2024-03-05.csv",
    PROJECT_ROOT + "/data/ETHBTC-trades-2024-03-05.csv",
]

engine = yabe.TriangularEngine(latency_ns=500_000, maker_fee_bps=-1.0, taker_fee_bps=7.5)
engine.enable_microstructure_recording(True, sampling_interval_ns=1_000_000_000)

result = engine.run(paths)
df = pd.DataFrame(engine.get_microstructure_dataframe())


In [6]:
# 1. Убираем "холодный старт" (фильтруем строки, где стакан еще не собрался с обеих сторон)
df_clean = df[(df['bid_price'] > 0) & (df['ask_price'] > 0)].copy()

# 2. Вытаскиваем BTC (asset_id = 0)
df_btc = df_clean[df_clean['asset_id'] == 0].copy()

# 3. Считаем микроструктурные фичи!
df_btc['mid_price'] = (df_btc['bid_price'] + df_btc['ask_price']) / 2
df_btc['spread'] = df_btc['ask_price'] - df_btc['bid_price']
df_btc['book_imbalance'] = df_btc['bid_qty'] / (df_btc['bid_qty'] + df_btc['ask_qty'])

# Выводим красоту
df_btc[['timestamp', 'mid_price', 'spread', 'book_imbalance']].head(10)

,timestamp,mid_price,spread,book_imbalance
15,1709596806115000000,68212.005,0.01,0.732953
23,1709596809172000000,68154.525,0.29,0.997723
27,1709596811289000000,68132.745,0.75,0.998277
30,1709596812291000000,68124.355,0.69,0.992198
32,1709596813297000000,68123.355,2.69,0.910937
37,1709596815440000000,68106.000,4.00,0.964216
40,1709596816528000000,68119.050,4.10,0.961842
42,1709596817545000000,68117.490,0.98,0.999578
43,1709596818558000000,68117.490,0.98,0.999477
45,1709596819571000000,68119.975,0.25,0.997990
